In [1]:
import pandas as pd
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.svm import SVC

# ==========================================
# 1. LOAD DATA & ENGINEER FEATURES
# ==========================================
df = pd.read_csv('cognitive_training_set.csv')
y = df['target']

# Feature Engineering (Must match your App exactly)
df['total_cog'] = df['mr_totalraw'] + df['voc_totalraw'] + df['lns_totalraw']
df['working_mem_ratio'] = df['lns_totalraw'] / (df['mr_totalraw'] + 1)

feature_cols = ['age', 'gender_encoded', 'mr_totalraw', 'voc_totalraw', 
                'lns_totalraw', 'total_cog', 'working_mem_ratio']
X = df[feature_cols]

# ==========================================
# 2. DEFINE THE 2-MODEL ENSEMBLE
# ==========================================
# We use the custom weights that gave us 81.11%
CUSTOM_WEIGHTS = {0: 1, 1: 1.5}

# Model 1: SVM (The Curve Finder)
model_svm = make_pipeline(
    StandardScaler(), 
    SVC(kernel='rbf', class_weight=CUSTOM_WEIGHTS, probability=True)
)

# Model 2: Random Forest (The Rule Maker)
model_rf = RandomForestClassifier(
    n_estimators=100, 
    class_weight=CUSTOM_WEIGHTS, 
    max_depth=5, 
    random_state=42
)

# The Voter
ensemble = VotingClassifier(
    estimators=[
        ('svm', model_svm),
        ('rf', model_rf)
    ],
    voting='soft'
)

# ==========================================
# 3. TRAIN & EXPORT
# ==========================================
print("⚙️ Training Final 2-Model Ensemble on full dataset...")
ensemble.fit(X, y)

filename = 'model_game_custom.pkl'
joblib.dump(ensemble, filename)

print(f"✅ SUCCESS: Saved '{filename}'")
print(f"   (Accuracy: ~81.11% | Models: SVM + Random Forest)")

⚙️ Training Final 2-Model Ensemble on full dataset...
✅ SUCCESS: Saved 'model_game_custom.pkl'
   (Accuracy: ~81.11% | Models: SVM + Random Forest)
